# Binary Search Trees

---

## The Big WHY

A regular binary tree has no rules about where values go. A **BST adds one rule** that changes everything:

> For every node: all values in the **left subtree are smaller**, all values in the **right subtree are larger**.

This one rule gives us O(log n) search — instead of checking every node like a plain tree, we eliminate half the tree at every step, exactly like binary search on a sorted array.

```
        8
       / \
      3   10
     / \    \
    1   6    14
       / \
      4   7
```

Search for 4:
- 4 < 8 → go left to 3
- 4 > 3 → go right to 6
- 4 < 6 → go left to 4 ✓

3 comparisons to find it in a 7-node tree. That's O(log n).

---

## BST Properties

| Property | Detail |
|----------|--------|
| Search | O(log n) average, O(n) worst (skewed tree) |
| Insert | O(log n) average |
| Delete | O(log n) average |
| Inorder traversal | Always gives sorted output |

**Worst case O(n):** If we insert values in sorted order (1, 2, 3, 4...), the BST degenerates into a linked list — every node only has a right child. This is why balanced BSTs (AVL, Red-Black trees) exist.

---

## ML Connection

**Decision Tree nodes are BST-like splits.** Each internal node asks `feature_value < threshold?` — left if yes, right if no. The tree is built to maximize information gain at each split, not to maintain BST order, but the traversal logic is identical.

**Why tree depth matters in ML:** A very deep decision tree = O(n) lookup in the worst case = overfitting. Sklearn's `max_depth` parameter prevents this — it's essentially capping the tree height to keep predictions fast and generalizable.

---

In [16]:
# Paste TreeNode, build_tree, tree_to_list from binary_trees.ipynb
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    
    def __repr__(self):
        return f"TreeNode({self.val})"
    
def build_tree(values):
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1

    while queue and i < len(values):
        node = queue.popleft()

        # left child
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        
        i += 1

        # right child
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i])
                queue.append(node.right)
        
        i += 1

    return root

def tree_to_list(root):
    queue = deque([root])
    result = []

    while queue:
        node = queue.popleft()

        if node is not None:
            result.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
        else:
            result.append(None)   # represent the missing nodes, part of the structure

         # remove the trailing nones because they are just empty padding
    while result and result[-1] is None:
        result.pop()
    return result

In [17]:
def inorder(node, result=None):
    if result is None:
        result = []
    if node is None:
        return result
    inorder(node.left, result)
    result.append(node.val)
    inorder(node.right, result)
    return result

## Part 1 — BST Search

Given a BST root and a target value, return the node if found, else `None`.

**The logic at each node:**
- target == node.val → found it
- target < node.val → go left
- target > node.val → go right

In [18]:
# bst_search(root, target) → TreeNode or None
# Base case: root is None → return None
# Compare target to root.val, recurse left or right accordingly

def bst_search(root, target):
    if root is None:
        return None
    
    if target == root.val:
        return root
    elif target < root.val:
        return bst_search(root.left, target)
    else:
        return bst_search(root.right, target)
# Build test BST:
#        8
#       / \
#      3   10
#     / \    \
#    1   6    14

root = build_tree([8, 3, 10, 1, 6, None, 14])

print(bst_search(root, 6))   # TreeNode(6)
print(bst_search(root, 14))  # TreeNode(14)
print(bst_search(root, 5))   # None


TreeNode(6)
TreeNode(14)
None


## Part 2 — BST Insert

Insert a new value into a BST. Always insert at a leaf — never restructure existing nodes.

**The logic:** Follow the same path as search. When you hit `None`, that's where the new node goes.

**Key insight:** Return the node at each step — this is how the parent gets connected to the newly inserted node.

In [19]:
# bst_insert(root, val) → root
# Base case: root is None → return TreeNode(val)  ← this is where we insert
# Compare val to root.val, recurse left or right
# Return root at the end

def bst_insert(root, val):
    # base case
    if root is None:
        return TreeNode(val)
    if val < root.val:
        root.left = bst_insert(root.left, val)
    else:
        root.right = bst_insert(root.right, val)

    return root
# Test — insert 5 into the BST above:
# inorder should give sorted output including 5
root = build_tree([8, 3, 10, 1, 6, None, 14])
bst_insert(root, 5)
print(inorder(root))  # [1, 3, 5, 6, 8, 10, 14]


[1, 3, 5, 6, 8, 10, 14]



```python
Why we return and assign back:

When you call bst_insert(root.left, val), you're passing node 3's left child. If that child is None, the function creates a new node and returns it — but node 3 doesn't know about it yet. You have to explicitly tell node 3:


root.left = bst_insert(root.left, val)
#  ↑ node 3 now points to the new node
Without the assignment, the new node is created and immediately lost — nothing in the tree points to it.
```

## Part 3 — BST Delete

Delete is the trickiest BST operation. Three cases:

**Case 1 — Node has no children (leaf):** Just remove it → return `None`

**Case 2 — Node has one child:** Replace it with its only child → return that child

**Case 3 — Node has two children:** Find the **inorder successor** (smallest value in right subtree), copy its value to current node, then delete the inorder successor from the right subtree.

```
Delete 3 from:
        8
       / \
      3   10
     / \
    1   6

Inorder successor of 3 = smallest in right subtree = 6
Copy 6 into node → delete 6 from right subtree

Result:
        8
       / \
      6   10
     /
    1
```

In [23]:
# bst_delete(root, val) → root
# Step 1: find the node (same as search — recurse left/right)
# Step 2: handle the 3 cases above
# Helper: find the minimum node in a subtree (go left until None)
def bst_delete(root, val):
    if root is None:
        return None
    if val < root.val:
        root.left = bst_delete(root.left, val)
    elif val > root.val:
        root.right = bst_delete(root.right, val)
    else:
        # found the node so we handle the 3 cases here
        if not root.left and not root.right:
            return None
        if root.left is None:
            return root.right
        if root.right is None:
            return root.left
        
        # now we find smallest in right subtree
        successor = root.right
        while successor.left:
            successor = successor.left
        
        # we copy its value, then delete it from right subtree
        root.val = successor.val
        root.right = bst_delete(root.right, successor.val)
    return root

            

# Test 
root = build_tree([8, 3, 10, 1, 6, None, 14])
bst_delete(root, 3)
print(inorder(root))  # [1, 6, 8, 10, 14]


[1, 6, 8, 10, 14]


---

## LeetCode

### Problem 1: Validate BST (LC #98)

**What it asks:** Given a binary tree, return `True` if it is a valid BST.

**The trap:** Checking `node.left.val < node.val < node.right.val` at each node is NOT enough.

```
    5
   / \
  1   4
     / \
    3   6
```
Node 4's children look fine locally (3 < 4 < 6), but 4 is in the right subtree of 5 — it must be > 5. It's not. Invalid BST.

**The fix:** Pass down `min_val` and `max_val` bounds. Every node must satisfy `min_val < node.val < max_val`.

- Root starts with bounds `(-inf, +inf)`
- Going left: update max_val to root.val
- Going right: update min_val to root.val

In [24]:
# isValidBST(root, min_val=-inf, max_val=+inf) → bool
# Base case: None → True
# Check: min_val < root.val < max_val, else False
# Recurse left with updated max_val, recurse right with updated min_val

def isValidBST(root, min_val = float("-inf"), max_val = float("inf")):
    if root is None: # base case
        return True
    
    if not (min_val < root.val < max_val):
        return False
    left_valid = isValidBST(root.left, min_val, root.val)
    right_valid = isValidBST(root.right, root.val, max_val)
    return left_valid and right_valid
# Tests:
print(isValidBST(build_tree([2, 1, 3])))                        # True
print(isValidBST(build_tree([5, 1, 4, None, None, 3, 6])))      # False
print(isValidBST(build_tree([2, 2, 2])))                        # False


True
False
False


### Problem 2: Kth Smallest Element in BST (LC #230)

**What it asks:** Find the kth smallest value in a BST.

**The key insight:** Inorder traversal of a BST gives values in sorted order. So the kth smallest is just the kth element in the inorder traversal.

Two approaches:
1. Collect full inorder list → return `result[k-1]`
2. Count nodes during inorder traversal, stop at k (more efficient)

Try approach 1 first, then approach 2 if you have time.

In [26]:
# kthSmallest(root, k) → int
# Approach 1: inorder traversal → return element at index k-1
def kthSmallest(root, k):
    result = []
    def inorder(node):
        # base case
        if node is None:
            return None
        inorder(node.left)
        result.append(node.val)
        inorder(node.right)
    inorder(root)
    return result[k-1]

# approach 2: we count as we traverse and stop when we hit the kth node
def kthSmallestApproach2(root, k):
    count = [0]    # Since integers aren't mutable in Python, use a list of one element to share state
    result = [None] # stores the answer once found

    def inorder(node):
        if node is None:
            return
        inorder(node.left)          # go left first (smallest values)
        count[0] += 1               # visited one more node
        if count[0] == k:           # found the kth smallest
            result[0] = node.val
            return
        inorder(node.right)         # go right (larger values)

    inorder(root)
    return result[0]

# Tests:
print(kthSmallest(build_tree([3, 1, 4, None, 2]), 1))                        # 1
print(kthSmallestApproach2(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3))            # 3


1
3


---

## Summary

| Concept | Key idea |
|---------|----------|
| BST rule | Left < Node < Right, at every node |
| Search | Eliminate half the tree each step — O(log n) |
| Insert | Follow search path, insert at the None you hit |
| Delete | 3 cases: leaf, one child, two children (inorder successor) |
| Validate BST | Pass min/max bounds down — local check is not enough |
| Kth smallest | Inorder traversal gives sorted order |
